In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FastFood-Partie1-Validation")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "2")
    .config("spark.cores.max", "4")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

In [ ]:
print("Spark version:", spark.version)
print("Master:", sc.master)
print("Application:", sc.appName)
print("Application ID:", sc.applicationId)
print("Default parallelism:", sc.defaultParallelism)

print(
    "Hadoop version:",
    sc._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

In [ ]:
rdd = sc.parallelize(range(1, 1001), 4)

result = rdd.sum()

print("Partitions:", rdd.getNumPartitions())
print("Result:", result)

assert result == 500500

print("RDD TEST PASSED")

In [ ]:
def inspect_partition(index, iterator):
    import socket

    values = list(iterator)

    yield {
        "partition": index,
        "worker": socket.gethostname(),
        "records": len(values),
        "sum": sum(values)
    }

results = rdd.mapPartitionsWithIndex(
    inspect_partition
).collect()

for result in results:
    print(result)

In [ ]:
from src.upload_data import s3, BUCKET

buckets = {bucket["Name"] for bucket in s3.list_buckets()["Buckets"]}
if BUCKET not in buckets:
    s3.create_bucket(Bucket=BUCKET)
s3.head_bucket(Bucket=BUCKET)
print("MINIO TEST PASSED")


In [ ]:
import os
from kafka.admin import KafkaAdminClient

admin = KafkaAdminClient(
    bootstrap_servers=os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:29092"),
    request_timeout_ms=10000,
)
try:
    topics = admin.list_topics()
    assert "orders" in topics, topics
finally:
    admin.close()
print("KAFKA TEST PASSED: orders")

In [ ]:
spark.stop()